# Phase 1: Multi-prototype study

## Section 1 - Purpose and overview

This study asks whether representing a class with more than one prototype corrects the known limits of perception: the empty mouth slot predicted as mouth, the loose mouth seen from the side, and the identity of the two arms, without making the rest worse.

The current method represents each class with one prototype, the normalized mean of its annotated patches (Snell et al. 2017, Sect. 2.2). A diagnostic on the reference frames showed why a mean can fail. The empty mouth slot is annotated, inside the body polygon of 3_disassembled_ordered_f0001, but it is one patch out of 251 body patches, and that patch is closer to the mouth prototype than to the body prototype. An example can be present and still be lost in the mean.

Two changes are made, in this order: first the examples, then how they are used.

The examples follow a coverage rule: for each piece, the support set contains at least one example of each state in which the piece appears during the assembly. This is our own design rule. It is in the spirit of the viewpoint annotation of PASCAL VOC, introduced for methods that treat different viewpoints separately (Everingham et al. 2010, Sect. 3.3). Two states are missing from the current support set, and one frame is added for each, from reference recordings only. caos_f0043 adds the loose mouth seen from the side (use_labels: mouth). 4_partially_assembled_f0002 adds the body on the base with the face not yet complete, including the empty mouth slot, and a loose mouth seen at three quarters (use_labels: body, mouth). All visible pieces are annotated in both frames, so that unused pieces become ignored patches and not background. 4_partially_assembled_f0002 was a candidate in the support-set study and was not selected there; it enters now because of the coverage rule, not because of a score. The support set of 10 frames is adopted by rule, and its comparison with the support set of 8 frames is reported only as a description.

How the examples are used is the question of the study. Mensink et al. (2013, Sect. 1) contrast two distance-based classifiers: the nearest class mean, which represents a class by its mean, and the k-nearest neighbour, which uses all its examples. They also extend the first to several centroids per class (Sect. 3.3). Three variants follow from this. V0 is the current method, one prototype per class. V1 has one prototype per class and aspect: a piece instance is mounted if it touches the figure, loose otherwise, and background and hand keep a single prototype. This is the idea of several centroids per class, with the groups given by the aspect rather than by k-means, because the slot and the side view of the mouth cover one to three patches and could not form a cluster of their own. On unit vectors it corresponds to a mixture of von Mises-Fisher distributions (Banerjee et al. 2005, Sect. 3). V2 uses every annotated patch as a prototype, the nearest neighbour. In all variants the similarity of a class is the highest similarity among its prototypes, and the background margin is applied as before.

The hypotheses are written before running. In V0 the two new frames change little, because their few patches are lost in the means, as in the support-set study. V2 should predict the empty slot as body, because the annotated slot patches become body prototypes; the risk is that body patches next to the face, which already look like face pieces, take patches from the face pieces. V1 probably does not correct the slot, which stays a few patches in the mean of its aspect. The side view of the mouth keeps one patch after labelling, so only V2 can use it. No variant should correct the identity of the arms along their stick, where the patches of the two arms look the same. Mensink et al. found the nearest class mean better than the nearest neighbour in their setting, so V2 may also be worse overall.

The decision rule is the one of the support-set study. A variant replaces V0 only if the paired 95% bootstrap interval of the mIoU difference, with the arms merged and background margin 0.03, lies entirely above zero (Koehn 2004, Sect. 6). If both variants pass, V1 is chosen, being simpler and faster, unless V2 is better than V1 by the same criterion. Three targeted measures are reported but do not enter the rule: body patches predicted as mouth in the frames where the mouth is not mounted, the share of arm patches given the right arm, loose and mounted, and the mouth IoU in the two validation frames of 1_disassembled_controlled_caos. The time of classification is also measured, since V2 compares each patch with tens of thousands of prototypes.

The two new frames were motivated by errors seen on validation, and the variants are chosen on validation, which makes the results optimistic (Cawley and Talbot 2010, Sect. 4.4 and 5). For this reason the variants are few and fixed here, and the test sequence (assembly_1) is not read.

Section 2 sets the configuration. Section 3 checks that the new code reproduces the current results. Section 4 builds the variants, Section 5 compares them, Section 6 reports the targeted measures, Section 7 the time, and Section 8 applies the rule.

## Section 2 - Configuration

All paths, frames and hyperparameters in one place. The values are those of the pipeline (Section 2.5) and of the support-set study (Section 2.4). The support set has two more frames, added by the coverage rule (Section 1).

In [ ]:
# 2.1 Dependencies: install once per environment with pip install -r requirements.txt (README, Setup)

In [ ]:
# 2.2 Imports
%load_ext autoreload
%autoreload 2

import sys
import json
import time
import hashlib
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy import ndimage
import transformers
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches